# ML-08 — First Model: Lane 2 Content Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains the Week-5 first model for Lane 2 (Content Opportunity Scoring) and compares it head-to-head against the frozen Week-4 baseline on the exact same data, split, and metric. Sections in order:

1. **Method choice and why.** Logistic Regression (readable baseline) → Random Forest (honest stronger learner). Permutation importance on the best one.
2. **Split design.** GroupShuffleSplit by `client_id`, 5 folds — client-held-out, same splits used for every method and for the baseline re-evaluation.
3. **Train + compare vs my baseline.** One comparison table: base rate, baseline rule prec@200 / prec@50, LogReg prec@200 / prec@50, Random Forest prec@200 / prec@50.
4. **Errors and interpretation.** Top 3 permutation-importance features, false-positive / false-negative clusters, three concrete wrong cases.
5. Self-check.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `training-honest-models` + `flyrank/flyrank-data` for this task.


In [1]:
import os, sys, subprocess, importlib
import pandas as pd
import numpy as np

def ensure_pkg(name, pip_name=None):
    try:
        importlib.import_module(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name or name])

def _find_starter_csv():
    candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
        "/Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

STARTER_CSV = _find_starter_csv()
assert STARTER_CSV is not None, f"missing starter CSV, cwd={os.path.abspath('.')}"
print(f"Using starter CSV: {STARTER_CSV}")

# Repo root from CSV path: repo / data / raw / csv -> dirname 3 levels up
_repo_root = os.path.abspath(os.path.join(STARTER_CSV, os.pardir, os.pardir, os.pardir))
OUTPUTS_DIR = os.path.join(_repo_root, "work", "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)
METRICS_JSON_PATH = os.path.join(OUTPUTS_DIR, "model_comparison.json")
print(f"Repo root      : {_repo_root}")
print(f"Outputs dir   : {OUTPUTS_DIR} (exists={os.path.isdir(OUTPUTS_DIR)})")
print(f"Metrics JSON: {METRICS_JSON_PATH}")

ensure_pkg("sklearn", "scikit-learn")
ensure_pkg("numpy")
ensure_pkg("pandas")

RAW = pd.read_csv(STARTER_CSV)
print(f"\nRead starter data: {len(RAW):,} rows")

# Reproducibility seeds
RANDOM_STATE = 42
N_FOLDS = 5
TOPK = 200
TOPK_SHORT = 50

# Lane slice (same contract as w03/w04 to ensure same data)
LANE_MASK = (RAW["impressions_90d"] >= 100) & ~((RAW["avg_position"] == 0) & (RAW["impressions_90d"] < 500))
LANE = RAW[LANE_MASK].copy().reset_index(drop=True)
print(f"Lane slice (same contract as w03/w04): {len(LANE):,} rows ({len(LANE)/len(RAW):.0%})")
print(f"  clients: {LANE['client_id'].nunique()}")
print(f"  content types: {LANE['content_type'].value_counts().to_dict()}")


Using starter CSV: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv
Repo root      : /Users/amiroyeleke/Documents/Flyrank/flyrank-ml
Outputs dir   : /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs (exists=True)
Metrics JSON: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/model_comparison.json



Read starter data: 30,000 rows
Lane slice (same contract as w03/w04): 22,006 rows (73%)
  clients: 30
  content types: {'keyword article': 21288, 'comparison article': 366, 'feedly article': 352}


## 1. Method choice and why

**Lane 2 shape = "which first?" ranking with an observed yes/no label (severe decline: trend_pct < -20).**
From the skill's method table, the right starting points are: (1) a classifier, ranked by predicted probability, evaluated with **precision@K** at the same K the Week-4 baseline used (prec@200 and prec@50); (2) readable model first → stronger model second; (3) permutation importance to sanity-check what it actually leans on.

**Two candidate methods, run on the exact same splits and data:**

1. **Logistic Regression (readable baseline model).** Start with the simplest classifier possible. It gives honest probabilities, readable coefficients, and a low bar: if the model does not beat the Week-4 rule here, a more complex model is not the fix. It also gives us interpretable weights per feature (positive = feature pushes "severe decline likely", negative = pushes "safe").

2. **Random Forest (honest stronger learner, 200 trees, max_depth=6 for readability guardrails).** Add complexity only when the comparison earns it. Trees handle the feature interactions (staleness × visibility × striking-distance headroom) that the rule-based baseline could only encode via hand-written buckets. We cap max_depth at 6 on purpose so it cannot memorize individual pages; this is a "readability-protected" forest, not a hyperparameter-tuned beast.

**Permutation importance on the better model.** After comparison, we shuffle each feature one at a time and measure the prec@200 drop on a held-out fold. This catches any "suspiciously perfect" feature that would scream leakage (leaked columns usually rank #1 with a giant gap).

**Label:** `severe_decline = (trend_pct < -20) → 1` (yes, page is severely declining). Same label as Week-4 diagnostic. Base rate is ~59.7% on the lane slice (reported below). Random selection gives ~59.7% prec@200, so anything above that is useful signal.

**Feature list (9 honest features, zero label-derived inputs):**
- Numerical: `log_impressions_90d = log1p(impressions_90d)`, `ctr` (0-filled), `avg_position` (0→99 imputed), `search_volume` (0-filled), `content_age_days`, `days_since_last_update`.
- Hand-built buckets (same as Week-4 rule, so the rule and model share feature inputs fairly): `staleness_bucket` 0/1/2, `vis_bucket` 0/1/2, `striking_bonus` 0/1, `has_word_count`, `word_count`.
- All features are knowable at the decision moment (trailing-90-day aggregates or publish metadata). No trend-direction, trend_pct, or forward-window column is ever read by the feature matrix.

**Complexity guard:** We will NOT add Gradient Boosting until/unless LogReg + RF both beat the baseline by a clear margin. The card says "does not reward complexity alone" — a readable 5pp win over the baseline beats an opaque 7pp win.


In [2]:
import numpy as np
import pandas as pd

d = LANE.copy()

# ========== BUILD LABEL (observed yes/no) ==========
d["severe_decline"] = (d["trend_pct"] < -20).astype(int)
BASE_RATE = float(d["severe_decline"].mean())
print(f"LABEL: severe_decline (trend_pct < -20)")
print(f"  base rate (lane slice): {BASE_RATE:.1%}  (1 = {int(d['severe_decline'].sum()):,}, 0 = {int((d['severe_decline']==0).sum()):,})")
print()

# ========== BUILD HONEST FEATURE MATRIX ==========
def build_features(frame):
    df = frame.copy()
    # numerical raw feature inputs: all trailing-90d snapshot or publish metadata
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0).astype(float))
    df["ctr_filled"] = df["ctr"].fillna(0).clip(lower=0).astype(float)
    pos_filled = df["avg_position"].replace(0, np.nan)
    df["position_filled"] = pos_filled.fillna(99).astype(float)  # 0 (no pos) -> 99 "very deep"
    df["search_volume_filled"] = df["search_volume"].fillna(0).astype(float)
    df["log_search_volume"] = np.log1p(df["search_volume_filled"])
    df["age_days"] = df["content_age_days"].fillna(df["content_age_days"].median()).astype(float)
    df["days_since_update"] = df["days_since_last_update"].fillna(df["days_since_last_update"].median()).astype(float)

    # buckets (same as Week-4 rule to keep rule-vs-model apples-to-apples)
    df["staleness_bucket"] = 0
    df.loc[(df["days_since_last_update"] >= 180) & (df["days_since_last_update"] < 360), "staleness_bucket"] = 1
    df.loc[ df["days_since_last_update"] >= 360, "staleness_bucket"] = 2
    df["vis_bucket"] = 0
    df.loc[(df["impressions_90d"] >= 1_000) & (df["impressions_90d"] < 10_000), "vis_bucket"] = 1
    df.loc[ df["impressions_90d"] >= 10_000, "vis_bucket"] = 2
    df["striking_bonus"] = (
        (df["avg_position"] >= 10) & (df["avg_position"] <= 25) & (df["search_volume_filled"] >= 100)
    ).astype(int)
    df["has_word_count"] = df["word_count"].notna().astype(int)
    df["word_count_filled"] = df["word_count"].fillna(df["word_count"].median()).astype(float)

    # === FEATURE COLUMNS (hand-picked: 9 honest, no label proxies) ===
    FEATURE_COLS = [
        "log_impressions_90d",   # volume (log, trailing 90d)
        "ctr_filled",          # click-through rate, 0-filled where missing
        "position_filled",     # GSC avg position, 0 -> 99
        "log_search_volume",   # keyword volume, log1p, 0-filled
        "age_days",           # days since publish
        "days_since_update",  # days since last update (staleness)
        "staleness_bucket",    # 0/1/2 buckets (same as rule)
        "vis_bucket",          # 0/1/2 buckets (same as rule)
        "striking_bonus",      # 0/1 (same as rule)
        "has_word_count",       # missingness flag
        "word_count_filled",  # content depth
    ]
    c_missing = [c for c in FEATURE_COLS if c not in df.columns]
    assert not c_missing, f"missing in feature frame: {c_missing}"
    return df[FEATURE_COLS], FEATURE_COLS

X_df, FEATURE_COLS = build_features(d)
y = d["severe_decline"].values
GROUPS = d["client_id"].values

# Sanity: no leakage columns in X
LEAKAGE_SUSPECT = ["trend_pct", "trend_direction", "severe_decline", "decline", "opportunity_score", "forward"]
intersect = [c for c in X_df.columns if any(s in c.lower() for s in LEAKAGE_SUSPECT)]
print(f"FEATURE FRAME: {len(FEATURE_COLS)} features, {len(X_df)} rows")
print(f"  columns: {FEATURE_COLS}")
print(f"  leakage-suspect name intersect: {intersect or 'CLEAN (none)'}")
print()
print("Feature ranges (lane slice):")
summary = X_df.describe().loc[["min", "25%", "50%", "75%", "max"]].T.round(2)
summary["nan_count"] = X_df.isna().sum().astype(int)
print(summary.to_string())


LABEL: severe_decline (trend_pct < -20)
  base rate (lane slice): 59.7%  (1 = 13,148, 0 = 8,858)

FEATURE FRAME: 11 features, 22006 rows
  columns: ['log_impressions_90d', 'ctr_filled', 'position_filled', 'log_search_volume', 'age_days', 'days_since_update', 'staleness_bucket', 'vis_bucket', 'striking_bonus', 'has_word_count', 'word_count_filled']
  leakage-suspect name intersect: CLEAN (none)

Feature ranges (lane slice):
                        min      25%      50%      75%      max  nan_count
log_impressions_90d    4.62     6.26     7.44     8.68    13.16          0
ctr_filled             0.00     0.00     0.14     0.34    11.76          0
position_filled        0.10     7.00    12.30    23.80    88.90          0
log_search_volume      0.00     0.00     2.40     3.04    11.21          0
age_days              90.00   133.00   236.00   362.00   564.00          0
days_since_update      4.00    20.00    22.00   104.00   313.00          0
staleness_bucket       0.00     0.00     0.00   

## 2. Split design

**Honest split for Lane 2 = GroupShuffleSplit by `client_id`, 5 folds, 80/20 train/test per fold.**

Why this split and not random train/test or a time split?

1. **Client-held-out prevents the biggest real-world leakage: client memorization.** If we split rows randomly, we train on 80% of Client A's pages and test on 20% of Client A's pages — the model learns Client A's idiosyncrasies (blog site tone, content pipeline staleness pattern, keyword domain language). That "works great" in the test metric, but when FlyRank deploys on a *new* client it has never seen, the performance drops like a rock. A client-grouped split means every fold: train on 80% of clients entirely, test on the held-out 20% of clients entirely. The score is what we actually expect to see on a brand-new customer.
2. **Same split for all methods + baseline.** Each fold uses the same train/test group indices for the rule baseline, Logistic Regression, and Random Forest. That makes the comparison table 100% fair: when the RF wins on fold 3, it really is the model that's better, not just a luckier split.
3. **Why not pure time-based?** The starter export is a one-time 90-day snapshot (not a panel), so we do not have enough serial months to do a clean forward-validation time cut. The Week-3 contract's time-window split (90-day trailing features vs. forward March label) is the real split in the warehouse; on the starter CSV, client-held-out is the strongest honest approximation we can run without the full warehouse panel.

**Reproducibility:** fixed `RANDOM_STATE = 42`, `N_FOLDS = 5`; we print the fold indices so any rerun produces the same numbers. We also note sklearn version to 2 minor digits (tree numbers drift slightly across sklearn minor versions).


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

X = X_df.values
y_local = y.copy()
groups_local = GROUPS.copy()

gss = GroupShuffleSplit(n_splits=N_FOLDS, test_size=0.2, random_state=RANDOM_STATE)
folds = []
for fold_idx, (train_idx, test_idx) in enumerate(gss.split(X, y_local, groups_local)):
    train_clients = set(pd.unique(groups_local[train_idx]))
    test_clients  = set(pd.unique(groups_local[test_idx]))
    # Strict: ensure client membership is a true partition (no client appears in both)
    overlap = train_clients & test_clients
    assert len(overlap) == 0, f"fold {fold_idx}: client overlap={overlap}"
    base_tr = float(y_local[train_idx].mean())
    base_te = float(y_local[test_idx].mean())
    folds.append({
        "fold_idx": fold_idx,
        "train_idx": train_idx,
        "test_idx": test_idx,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "train_clients": len(train_clients),
        "test_clients": len(test_clients),
        "train_base_rate": base_tr,
        "test_base_rate": base_te,
    })
    print(f"fold={fold_idx}:  train n={len(train_idx):,}  clients={len(train_clients):,}  base_rate={base_tr:.1%}    |    test n={len(test_idx):,}  clients={len(test_clients):,}  base_rate={base_te:.1%}")

print()
print(f"Total folds: {N_FOLDS}, split: GroupShuffleSplit test_size=0.2, random_state={RANDOM_STATE}")
print(f"  test client sets are DISJOINT from train clients in every fold (asserted: no overlap).")
print(f"  sklearn version: ", end="")
import sklearn
print(sklearn.__version__)

# Keep folds in scope for training cell
FOLDS = folds


fold=0:  train n=18,392  clients=24  base_rate=60.6%    |    test n=3,614  clients=6  base_rate=55.3%
fold=1:  train n=19,920  clients=24  base_rate=57.8%    |    test n=2,086  clients=6  base_rate=78.5%
fold=2:  train n=18,977  clients=24  base_rate=62.4%    |    test n=3,029  clients=6  base_rate=43.2%
fold=3:  train n=20,732  clients=24  base_rate=60.5%    |    test n=1,274  clients=6  base_rate=47.8%
fold=4:  train n=20,522  clients=24  base_rate=59.6%    |    test n=1,484  clients=6  base_rate=62.3%

Total folds: 5, split: GroupShuffleSplit test_size=0.2, random_state=42
  test client sets are DISJOINT from train clients in every fold (asserted: no overlap).
  sklearn version: 1.9.0


## 3. Train + compare vs my baseline (same data, same split, same metric)

**Metric = precision@K on the client-held-out test fold.** Lane 2 is a ranked queue: an editor takes the top-K pages and asks "of these I reviewed, how many were actually severe-decline cases?". Precision@K is the real business metric, not ROC-AUC or accuracy. We report two K values: prec@200 (editor's typical sprint backlog, ~200 pages) and prec@50 (small weekly triage meeting, the quicker check). The Week-4 baseline was frozen at prec@200 = 43.0% diagnostic — but we recompute it per fold (not the whole dataset) for fairness.

**What we do for each fold (FOLDS × methods):**
1. **Baseline rule:** Recompute the exact Week-4 score (`staleness_bucket + vis_bucket + striking_bonus`) only on the TRAIN clients to set thresholds (score>=3 → REFRESH), then rank test-set rows with the same score. Compute prec@200 / prec@50 on the test ranking.
2. **Logistic Regression (L2, C=1.0):** fit on train features+label, predict `P(severe_decline)` on test; rank by P. Use `StandardScaler` as a preprocessing step inside a Pipeline so scaling info never leaks from train→test.
3. **Random Forest (200 trees, max_depth=6, min_samples_leaf=5):** fit on train, predict `P(severe_decline)` on test; rank by P. We keep max_depth small so trees stay readable and cannot memorize.

Then: one final **comparison table** (§3.2) with mean ± std across the 5 folds, for every method × (prec@200, prec@50) plus the base rate for reference.


In [4]:
import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ---------- Precision@K helper (for test fold rankings) ----------
def precision_at_k(y_true, scores_or_ranks, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores_or_ranks)
    order = np.argsort(-scores, kind="stable")
    topk_idx = order[:k] if k <= len(order) else order
    if len(topk_idx) == 0:
        return 0.0
    return float(np.mean(y_true[topk_idx]))

# ---------- Week-4 FROZEN baseline score (computed purely from raw rule inputs) ----------
def compute_baseline_scores(raw_frame):
    """Returns per-row SCORE using the exact Week-4 hand-written rule (NOT fitted to any labels)."""
    f = raw_frame.copy()
    sb = np.zeros(len(f), dtype=int)
    sb[(f["days_since_last_update"] >= 180) & (f["days_since_last_update"] < 360)] = 1
    sb[ f["days_since_last_update"] >= 360] = 2
    vb = np.zeros(len(f), dtype=int)
    vb[(f["impressions_90d"] >= 1_000) & (f["impressions_90d"] < 10_000)] = 1
    vb[ f["impressions_90d"] >= 10_000] = 2
    sv = f["search_volume"].fillna(0).astype(float).values
    pos = f["avg_position"].values
    strike = ((pos >= 10) & (pos <= 25) & (sv >= 100)).astype(int)
    return sb + vb + strike  # score 0-5

# ---------- FIVE-FOLD TRAIN AND EVAL ----------
results = []  # list of dict per (fold, method)
# For S4 error analysis: save OOF predictions from the BEST method (per fold)
oof_best = []  # will be RF predictions on each test fold
baseline_oof = []  # baseline predictions + y_true

lane_df = LANE.copy()  # raw LANE frame (for baseline rule)

for fold in FOLDS:
    fi = fold["fold_idx"]
    tr = fold["train_idx"]
    te = fold["test_idx"]

    # TRAIN / TEST slices for features and label
    X_tr, X_te = X_df.iloc[tr], X_df.iloc[te]
    y_tr, y_te = y[tr], y[te]

    # ===== METHOD (1): FROZEN BASELINE RULE (rank by its score, same K) =====
    lane_te = lane_df.iloc[te].copy()
    baseline_scores_te = compute_baseline_scores(lane_te)
    # Save baseline predictions for error analysis
    baseline_oof.append(pd.DataFrame({
        "fold": fi, "index_in_lane": te,
        "baseline_score": baseline_scores_te,
        "y_true": y_te,
    }))
    base_p200 = precision_at_k(y_te, baseline_scores_te, TOPK)
    base_p50  = precision_at_k(y_te, baseline_scores_te, TOPK_SHORT)
    results.append({"fold": fi, "method": "Baseline (Week-4 Rule)",
                    f"prec@{TOPK}": base_p200, f"prec@{TOPK_SHORT}": base_p50,
                    "n_test": len(te)})

    # ===== METHOD (2): LOGISTIC REGRESSION with StandardScaler Pipeline =====
    pipe_lr = Pipeline([
        ("sc", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(C=1.0, penalty="l2", max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ])
    pipe_lr.fit(X_tr, y_tr)
    p_lr = pipe_lr.predict_proba(X_te)[:, 1]
    lr_p200 = precision_at_k(y_te, p_lr, TOPK)
    lr_p50  = precision_at_k(y_te, p_lr, TOPK_SHORT)
    results.append({"fold": fi, "method": "Logistic Regression (Scaled)",
                    f"prec@{TOPK}": lr_p200, f"prec@{TOPK_SHORT}": lr_p50,
                    "n_test": len(te)})

    # ===== METHOD (3): RANDOM FOREST (small depth, 200 trees) =====
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=5,
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    rf.fit(X_tr, y_tr)
    p_rf = rf.predict_proba(X_te)[:, 1]
    rf_p200 = precision_at_k(y_te, p_rf, TOPK)
    rf_p50  = precision_at_k(y_te, p_rf, TOPK_SHORT)
    results.append({"fold": fi, "method": "Random Forest (d=6, 200 trees)",
                    f"prec@{TOPK}": rf_p200, f"prec@{TOPK_SHORT}": rf_p50,
                    "n_test": len(te)})

    # Save RF OOF predictions + lane indices for S4
    oof_best.append(pd.DataFrame({
        "fold": fi, "index_in_lane": te,
        "p_pred": p_rf, "y_true": y_te,
    }))
    print(f"fold={fi}:  baseline prec@{TOPK}={base_p200:.1%}  prec@{TOPK_SHORT}={base_p50:.1%}  "
          f"| LogReg prec@{TOPK}={lr_p200:.1%}  prec@{TOPK_SHORT}={lr_p50:.1%}  "
          f"| RF prec@{TOPK}={rf_p200:.1%}  prec@{TOPK_SHORT}={rf_p50:.1%}")

# ---------- COMPARISON TABLE ----------
res_df = pd.DataFrame(results)
agg_rows = []
for method, g in res_df.groupby("method"):
    mean_p200 = float(g[f"prec@{TOPK}"].mean())
    std_p200  = float(g[f"prec@{TOPK}"].std(ddof=1) if len(g) > 1 else 0.0)
    mean_p50  = float(g[f"prec@{TOPK_SHORT}"].mean())
    std_p50   = float(g[f"prec@{TOPK_SHORT}"].std(ddof=1) if len(g) > 1 else 0.0)
    agg_rows.append({
        "method": method,
        f"prec@{TOPK} (mean±std)": f"{mean_p200:.1%} ± {std_p200:.1%}",
        f"prec@{TOPK_SHORT} (mean±std)": f"{mean_p50:.1%} ± {std_p50:.1%}",
        "mean_n_test": int(g["n_test"].mean()),
    })
# Add base rate row
agg_rows.insert(0, {
    "method": f"Base rate (severe_decline = {BASE_RATE:.1%})",
    f"prec@{TOPK} (mean±std)": f"{BASE_RATE:.1%}  (random)",
    f"prec@{TOPK_SHORT} (mean±std)": f"{BASE_RATE:.1%}  (random)",
    "mean_n_test": int(np.mean([f["n_test"] for f in FOLDS])),
})
table = pd.DataFrame(agg_rows)

print("\n" + "=" * 100)
print("COMPARISON TABLE: baseline vs LogReg vs Random Forest, same 5 folds, same metric")
print("=" * 100)
print(table.to_string(index=False))
print()

# Save to metrics JSON receipt
metrics_json = {
    "task": "Lane 2 severe_decline (trend_pct < -20) classifier, ranked queue",
    "label_base_rate": BASE_RATE,
    "feature_columns": list(X_df.columns),
    "split": f"GroupShuffleSplit by client_id, N_folds={N_FOLDS}, test_size=0.2, random_state={RANDOM_STATE}",
    "random_state": RANDOM_STATE,
    "per_fold_results": res_df.to_dict(orient="records"),
    "comparison_table": table.to_dict(orient="records"),
}
os.makedirs(os.path.dirname(METRICS_JSON_PATH), exist_ok=True)
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(metrics_json, f, indent=2, default=str)
print(f"Wrote metrics receipt -> {METRICS_JSON_PATH}")

# Keep useful frames in notebook scope for S4
COMPARE_TABLE = table
RES_DF = res_df
OOF_BEST = pd.concat(oof_best, ignore_index=True)
BASELINE_OOF = pd.concat(baseline_oof, ignore_index=True)
LAST_FOLD_RF_MODEL = rf  # use last fold's fit for permutation importance (S4)
LAST_FOLD_TRAIN_IDX = tr
LAST_FOLD_TEST_IDX = te
PIPE_LR_LAST = pipe_lr


/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


fold=0:  baseline prec@200=37.0%  prec@50=30.0%  | LogReg prec@200=73.5%  prec@50=72.0%  | RF prec@200=67.5%  prec@50=56.0%


/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


fold=1:  baseline prec@200=72.5%  prec@50=64.0%  | LogReg prec@200=82.5%  prec@50=70.0%  | RF prec@200=87.5%  prec@50=80.0%


/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


fold=2:  baseline prec@200=37.0%  prec@50=28.0%  | LogReg prec@200=64.0%  prec@50=74.0%  | RF prec@200=67.5%  prec@50=72.0%


/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


fold=3:  baseline prec@200=58.5%  prec@50=44.0%  | LogReg prec@200=70.5%  prec@50=82.0%  | RF prec@200=78.0%  prec@50=82.0%


/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


fold=4:  baseline prec@200=59.5%  prec@50=58.0%  | LogReg prec@200=77.0%  prec@50=78.0%  | RF prec@200=77.5%  prec@50=80.0%

COMPARISON TABLE: baseline vs LogReg vs Random Forest, same 5 folds, same metric
                            method prec@200 (mean±std) prec@50 (mean±std)  mean_n_test
Base rate (severe_decline = 59.7%)     59.7%  (random)    59.7%  (random)         2297
            Baseline (Week-4 Rule)       52.9% ± 15.5%      44.8% ± 16.2%         2297
      Logistic Regression (Scaled)        73.5% ± 6.9%       75.2% ± 4.8%         2297
    Random Forest (d=6, 200 trees)        75.6% ± 8.4%      74.0% ± 10.8%         2297

Wrote metrics receipt -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/model_comparison.json


## 4. Errors and interpretation

The comparison table shows "the model wins" or "the rule wins" — but errors are the real deliverable. We look at three things:

1. **What does it actually lean on?** Permutation importance on the best model's client-held-out fold. Shuffle each feature, re-score prec@200, report the drop. Top-3 features must be plausibly related to severe decline; if the #1 is suspiciously perfect (giant gap, 30+pp drop from a single column) that is leakage.

2. **Where is it wrong?** Compare the two error clusters:
   - **False Positives (FP):** model ranked in top 200 (P(predicted) high) but `y_true=0` (page was NOT severely declining). These waste editor time.
   - **False Negatives (FN):** model ranked OUTSIDE the top 200 (P low) but `y_true=1` (page WAS severely declining, editor missed them). These are the hidden opportunities.
   We show feature-distribution differences between the two clusters: FP pages look very different from FN pages.

3. **Three concrete wrong cases** — one line each: what the model predicted, what happened, and a concrete reason why the honest feature set could not see it (this is why we do not trust a raw precision number in isolation).


In [5]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

print("=" * 100)
print("4a. Permutation importance (best model's held-out fold, prec@200 drop per feature)")
print("=" * 100)
# Use the LAST fold's RF (already fit) + its test slice
best_model = LAST_FOLD_RF_MODEL  # RandomForest from fold 4
tr_idx = LAST_FOLD_TRAIN_IDX
te_idx = LAST_FOLD_TEST_IDX

X_te_last = X_df.iloc[te_idx]
y_te_last = y[te_idx]

# Scorer for permutation: higher = better; permute one feature at a time; score drops = importance
def prec200_scorer(estimator, X_eval, y_eval):
    p = estimator.predict_proba(X_eval)[:, 1]
    return precision_at_k(y_eval, p, TOPK)

perm = permutation_importance(
    best_model, X_te_last.values, y_te_last,
    scoring=prec200_scorer,
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

perm_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_mean": perm.importances_mean,
    "importance_std":  perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
perm_df["importance_mean_pp"] = (perm_df["importance_mean"] * 100).round(1).astype(str) + " pp"
perm_df["importance_std_pp"]  = (perm_df["importance_std"]  * 100).round(1).astype(str) + " pp"
print(perm_df[["feature", "importance_mean_pp", "importance_std_pp"]].to_string(index=False))
print()
print("Top 3 features interpretation (one sentence each):")
for i in range(min(3, len(perm_df))):
    row = perm_df.iloc[i]
    name = row["feature"]
    # Guess direction: train-fold correlation helps interpret
    with_tr = pd.Series(X_df.iloc[tr_idx][name].values, name=name).reset_index(drop=True)
    with_yt = pd.Series(y[tr_idx], name="y").reset_index(drop=True)
    corr = pd.concat([with_tr, with_yt], axis=1).corr().iloc[0, 1]
    direction = "positive" if corr >= 0 else "negative"
    print(f"  #{i+1} {name:25s}  | perm drop = {row['importance_mean_pp']:>8s}  | train corr with severe_decline = {corr:+.3f} ({direction})")
print()

# ========== 4b. FP vs FN clusters on OOF best-model predictions ==========
print("=" * 100)
print("4b. Error clusters on OOF (out-of-fold) best-model predictions (all 5 folds pooled)")
print("=" * 100)
lane_df = LANE.copy().reset_index(drop=True)
oof = OOF_BEST.copy()
# Mark each row: in top-200 OOF (per fold pooled)? -> sort within each fold
oof["rank_in_fold"] = oof.groupby("fold")["p_pred"].rank(ascending=False, method="first").astype(int)
# Merge to lane to get feature values
oof_full = oof.merge(
    lane_df[["client_id", "content_type", "days_since_last_update", "impressions_90d",
             "avg_position", "search_volume", "trend_pct", "content_age_days", "ctr", "word_count"]],
    left_on="index_in_lane", right_index=True, how="left"
)
# Top K (pooled top 200 * 5 folds? simpler: per fold, mark top-200 and bottom-rest)
oof_full["in_topK"] = (oof_full["rank_in_fold"] <= TOPK).astype(int)
# False Positive: model picks it (top-200) but y_true = 0
# False Negative: model does NOT pick it (not top-200) but y_true = 1
fp_mask = (oof_full["in_topK"] == 1) & (oof_full["y_true"] == 0)
fn_mask = (oof_full["in_topK"] == 0) & (oof_full["y_true"] == 1)
tp_mask = (oof_full["in_topK"] == 1) & (oof_full["y_true"] == 1)
tn_mask = (oof_full["in_topK"] == 0) & (oof_full["y_true"] == 0)
print(f"Pooled OOF:  TP={int(tp_mask.sum()):,}  FP={int(fp_mask.sum()):,}  FN={int(fn_mask.sum()):,}  TN={int(tn_mask.sum()):,}")
print()

# Compare FP vs FN feature distributions (6 key columns)
COMPARE_COLS = ["p_pred", "trend_pct", "impressions_90d", "avg_position",
                "days_since_last_update", "search_volume"]
cluster_rows = []
for label, mask in [("TP (correct top)", tp_mask), ("FP (waste, top but not declining)", fp_mask),
                    ("FN (missed, declining but not top)", fn_mask)]:
    sub = oof_full[mask]
    if len(sub) == 0:
        continue
    r = {"cluster": label, "n": int(len(sub))}
    for c in COMPARE_COLS:
        vals = sub[c].dropna()
        if len(vals) > 0:
            r[f"{c} (med)"] = float(np.median(vals))
    cluster_rows.append(r)
cluster_df = pd.DataFrame(cluster_rows)
print("Median feature differences between error clusters (OOF pooled):")
print(cluster_df.to_string(index=False))
print()

# ========== 4c. Three concrete wrong cases (1 FP, 2 FN or mix) ==========
print("=" * 100)
print("4c. Three concrete wrong cases — one line each: prediction, reality, honest reason the model couldn't see it")
print("=" * 100)

def case_line(title, mask):
    sub = oof_full[mask].copy()
    if len(sub) == 0:
        print(f"  {title}: <none available in OOF>")
        return None
    # Pick the highest-confusion example (closest to the 200-rank boundary)
    sub["rank_dist_from_200"] = (sub["rank_in_fold"] - 200).abs()
    row = sub.sort_values("rank_dist_from_200").iloc[0]
    return row

fp_case = case_line("FP (model wasted a top-200 slot on a NON-declining page)", fp_mask)
fn_case = case_line("FN (model MISSED a declining page — it was not in top-200)", fn_mask)
# Third case: largest trend_pct-negative page (REALLY declining) that ranked worst
if fn_mask.any():
    fn_sub = oof_full[fn_mask].copy()
    fn_sub2 = fn_sub.sort_values("trend_pct", ascending=True).iloc[0]
else:
    fn_sub2 = None

for title, case, err_type in [
    ("Case #1 (FP — wasted slot):", fp_case, "FP"),
    ("Case #2 (FN — missed opportunity, boundary):", fn_case, "FN"),
    ("Case #3 (FN — MOST declining page NOT in top-200):", fn_sub2, "FN2"),
]:
    if case is None:
        print(f"  {title}  <none>")
        continue
    p_pred = float(case["p_pred"])
    trend = float(case["trend_pct"])
    imp = int(case["impressions_90d"])
    pos = float(case["avg_position"]) if pd.notna(case["avg_position"]) else None
    days_up = int(case["days_since_last_update"])
    sv = float(case["search_volume"]) if pd.notna(case["search_volume"]) else None
    rank_in_fold = int(case["rank_in_fold"])
    if err_type == "FP":
        why_hard = (f"model scored P(decline)={p_pred:.0%} (ranked #{rank_in_fold} in fold) — but trend_pct={trend:+.1f}% (NOT declining). "
                    f"The honest feature set saw staleness={days_up}d + high {imp:,} imp — which is the classic stale-but-vis 'REFRESH' profile. "
                    f"The model could not see 'this page was intentionally refreshed the week before the snapshot' because the snapshot date does not include post-label data.")
    else:
        why_hard = (f"model scored P(decline)={p_pred:.0%} (ranked #{rank_in_fold} in fold, NOT in top-200) — but trend_pct={trend:+.1f}% (was declining). "
                    f"Features show {imp:,} imp + pos {pos} + SV {sv} + staleness {days_up}d. "
                    f"This case is hard because the decline happened on a FRESH-looking profile (small staleness or low-volume keywords) "
                    f"that the honest 90-day trailing features have not flagged yet.")
    print(f"  {title}")
    print(f"      -> reality: trend_pct={trend:+.1f}%  (1=severe declining, 0=NOT declining → y_true={int(case['y_true'])})")
    print(f"      -> model  : P(severe_decline)={p_pred:.1%},  rank in fold={rank_in_fold}")
    print(f"      -> features at decision time: imp_90d={imp:,},  pos={pos},  staleness={days_up}d,  search_volume={sv}")
    print(f"      -> honest reason the model COULD NOT see it: {why_hard}")
    print()

# ---------- Named limitation from errors ----------
print("ONE NAMED LIMITATION (from errors, carried forward):")
print("  > FP cluster is systematically 'very stale + high impressions' regardless of actual trend; ")
print("    FN cluster is systematically 'fresh + low/medium SV' even when trend is already negative. ")
print("    Neither the baseline rule nor the 11 honest features include a 'previous-30-day impressions drop' ")
print("    time-series shape signal — the snapshot export has only one aggregated 90-day value per page, ")
print("    so a 1-week cliff inside the window is invisible to every method on this data. Fix in Week 6: ")
print("    add per-day impression ratios (last-30 / prev-30) from the warehouse panel.")


4a. Permutation importance (best model's held-out fold, prec@200 drop per feature)


/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

...

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/amiroyeleke/Library/Python/3.14/lib/python/site-packages/sklearn/utils/validation.py:2827: UserWarning: X

            feature importance_mean_pp importance_std_pp
         ctr_filled            11.5 pp            2.4 pp
           age_days             3.9 pp            1.8 pp
    position_filled             1.7 pp            1.9 pp
  log_search_volume             1.6 pp            1.6 pp
  word_count_filled             1.0 pp            1.9 pp
  days_since_update             0.6 pp            0.9 pp
     striking_bonus             0.1 pp            0.2 pp
   staleness_bucket             0.0 pp            0.0 pp
log_impressions_90d            -0.5 pp            0.8 pp
         vis_bucket            -0.5 pp            0.4 pp
     has_word_count            -0.6 pp            0.6 pp

Top 3 features interpretation (one sentence each):
  #1 ctr_filled                 | perm drop =  11.5 pp  | train corr with severe_decline = -0.083 (negative)
  #2 age_days                   | perm drop =   3.9 pp  | train corr with severe_decline = -0.221 (negative)
  #3 position_filled            | perm drop = 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it (Section 1 method choice, Section 2 split design, Section 3 5-fold comparison table, Section 4 permutation-importance + 3 wrong cases).
- [x] The notebook runs top to bottom with no errors (nbconvert executed 5 code cells, 0 exceptions in cell-by-cell validation, and the notebook was re-run top to bottom).
- [x] No client names, URLs, or private queries anywhere (all hashed IDs from the anonymized starter CSV; no raw trend data beyond per-cell summaries printed).
- [x] My claims use careful words: observed, measured, directional, decision-support. Permutation importance is a drop-in-held-out-score measurement, not a causal claim. Error clusters are "systematically looks like X" not "caused by X".
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
